In [1]:
import sys
import torch

sys.path.append('/app/dev')
from utils import clean_gpu_cache
from models.speechlms.salmonn7b.salmonn7b import Salmonn7BKVOpt, Salmonn7B

/usr/local/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



===================================BUG REPORT===================================
Welcome to bitsandbytes. For bug reports, please submit your error trace to: https://github.com/TimDettmers/bitsandbytes/issues
For effortless bug reporting copy-paste your error into this form: https://docs.google.com/forms/d/e/1FAIpQLScPB8emS3Thkp66nvqwmjTEgxp8Y9ufuWTzFyr9kJ5AoI47dQ/viewform?usp=sf_link
CUDA_SETUP: WARNING! libcudart.so not found in any environmental path. Searching /usr/local/cuda/lib64...
CUDA SETUP: Loading binary /usr/local/lib/python3.9/site-packages/bitsandbytes/libbitsandbytes_cpu.so...


/usr/local/lib/python3.9/site-packages/bitsandbytes/cuda_setup/paths.py:27: UserWarning: WARNING: The following directories listed in your path were found to be non-existent: {PosixPath('https'), PosixPath('//github.com/pypa/get-pip/raw/9af82b715db434abb94a0a6f3569f43e72157346/public/get-pip.py')}
  warn(
/usr/local/lib/python3.9/site-packages/bitsandbytes/cuda_setup/paths.py:27: UserWarning: WARNING: The following directories listed in your path were found to be non-existent: {PosixPath('{"userLocale"'), PosixPath('"en","osLocale"'), PosixPath('"en","defaultMessagesFile"'), PosixPath('"en","resolvedLanguage"'), PosixPath('{}}'), PosixPath('"/root/.vscode-server/bin/072586267e68ece9a47aa43f8c108e0dcbf44622/out/nls.messages.json","locale"'), PosixPath('"en","availableLanguages"')}
  warn(
/usr/local/lib/python3.9/site-packages/bitsandbytes/cuda_setup/paths.py:27: UserWarning: WARNING: The following directories listed in your path were found to be non-existent: {PosixPath('vs/workbench/a

In [1]:
import csv

def create_id_dict():
    csv_path = '/app/datasets/audioCaps/test.csv'
    id_dict = {}

    with open(csv_path, 'r', newline="") as f:
        reader = csv.reader(f)
        for row in reader:
            number_id = row[0]
            youtube_id = row[1]

            if youtube_id not in id_dict.keys():
                id_dict[youtube_id] = []

            id_dict[youtube_id].append(number_id)

    return id_dict


dict = create_id_dict()

In [2]:
dict['VjSEIRnLAh8']

['103542', '107015', '103624', '105563', '106974']

In [2]:
device_num = 4
clean_gpu_cache(device_num)
salmonn7b = Salmonn7BKVOpt(device_num=device_num, verbose=True)
# salmonn7b = Salmonn7B(device_num)

[GPU 4] allocated: 0.000 GB
[GPU 4] reserved : 0.000 GB


/usr/local/lib/python3.9/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 2/2 [00:10<00:00,  5.06s/it]
Some weights of LlamaForCausalLM were not initialized from the model checkpoint at lmsys/vicuna-7b-v1.5 and are newly initialized: ['reference_model.layers.28.mlp.gate_proj.weight', 'reference_model.layers.20.post_attention_layernorm.weight', 'reference_model.layers.16.mlp.down_proj.weight', 'reference_model.layers.23.self_attn.k_proj.weight', 'reference_model.layers.11.mlp.up_proj.weight', 'reference_model.layers.9.self_attn.o_proj.weight', 'reference_model.layers.15.self_attn.rotary_emb.inv_freq', 'reference_model.layers.7.self_attn.o_proj.weight', 'reference_model.layers.22.mlp.up_proj.weight', 'reference_model.layers.8.mlp.gate_proj.

Patched models.speechlms.salmonn7b.modeling_llama_kv_opt_salmonn
Patched LlamaMLP
Patched LlamaRMSNorm


In [3]:
wav_instruct_dict = {
    'water': {
        'wav': '/app/datasets/air_bench/Foundation/Sound_AQA_clothoaqa/river_mouth3.wav',
        'instruction': 'How many times exactly does the water splash?'
    },
    'cat_meowing': {
        'wav': '/app/datasets/air_bench/Foundation/Audio_Grounding_AudioGrounding/Y_GI7meqlYZk.flac',
        'instruction': 'How many dog barks there are in the audio?'
    },
    'transcribe': {
        'wav': '/app/datasets/librispeech/test-clean/237/126133/237-126133-0003.wav',
        'instruction': 'Transcribe the audio. Output only the raw transcription, no commentary.'
    },
    'gunshot':{
        'wav': '/app/datasets/air_bench/Foundation/Sound_AQA_avqa/3049.flac',
        'instruction': "how many gun shots there are in the audio?"
    },
    'police': {
        'wav': '/app/datasets/air_bench/Foundation/Sound_AQA_avqa/3593.flac',
        'instruction': "What are the police officers saying?"
    },

    # /app/datasets/air_bench/Foundation/Sound_AQA_avqa/3511.flac # water steam
    'song': {
        'wav': '/app/datasets/air_bench/Foundation/Music_Instruments_Classfication_MTJ-Jamendo/2971.mp3',
        'instruction': "What is the genere of the song in the audio?"
    },    
}

task = 'cat_meowing'

inputs = salmonn7b.get_inputs_for_forward(
            instruction=wav_instruct_dict[task]['instruction'],
            wav_path=wav_instruct_dict[task]['wav'],
            device_num=device_num
        )

In [5]:
# response = salmonn7b.generate(
#     inputs,
# )

response = salmonn7b.generate(
    **inputs,
    approach='opt',
    plot=False, 
    example=task, 
    opt_steps=1,
    opt_lr=7e-198, 
    lambda_kl=1
)

print(response)

modality_bos_idx: 8 | modality_eos_idx: 96

---------------------
Generation step: 0
Adam step: 0
student forward...
reference forward...
KL: 0.0002 | Audio Relevance: nan | Text Relevance: nan | Audio/Text:  nan | Overall loss: nan
λ_kl X kl: 0.0002 | λ_relevance_audio X audio_relevance: nan | λ_relevance_text X text_relevance: nan

---------------------
Generation step: 1
Adam step: 0
student forward...
reference forward...
KL: -0.0000 | Audio Relevance: nan | Text Relevance: nan | Audio/Text:  nan | Overall loss: nan
λ_kl X kl: -0.0000 | λ_relevance_audio X audio_relevance: nan | λ_relevance_text X text_relevance: nan

---------------------
Generation step: 2
Adam step: 0
student forward...
reference forward...
KL: -0.0002 | Audio Relevance: nan | Text Relevance: nan | Audio/Text:  nan | Overall loss: nan
λ_kl X kl: -0.0002 | λ_relevance_audio X audio_relevance: nan | λ_relevance_text X text_relevance: nan

---------------------
Generation step: 3
Adam step: 0
student forward...
ref